In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

train = pd.read_csv('/content/drive/My Drive/csv/Train.csv', usecols=["text", "target"])
test_df = pd.read_csv('/content/drive/My Drive/csv/Test.csv', usecols=["text"])

feat_cols = "text"
label_cols = "target"
train.head()

,text,target
0,The bitcoin halving is cancelled due to,1
1,MercyOfAllah In good times wrapped in its gran...,0
2,266 Days No Digital India No Murder of e learn...,1
3,India is likely to run out of the remaining RN...,1
4,In these tough times the best way to grow is t...,0


In [3]:
test_df.head()

,text
0,Why is explained in the video take a look
1,Ed Davey fasting for Ramadan No contest
2,Is Doja Cat good or do you just miss Nicki Minaj
3,How Boris Johnson s cheery wounded in action p...
4,Man it s terrible Not even a reason to get on ...


In [4]:
test_df[feat_cols] = test_df[feat_cols].astype(str)
train[feat_cols] = train[feat_cols].astype(str)

In [5]:
train['target'].isna().sum().sum()

np.int64(0)

In [6]:
train=train.dropna()

In [7]:
train.head()

,text,target
0,The bitcoin halving is cancelled due to,1
1,MercyOfAllah In good times wrapped in its gran...,0
2,266 Days No Digital India No Murder of e learn...,1
3,India is likely to run out of the remaining RN...,1
4,In these tough times the best way to grow is t...,0


In [8]:
test_df=test_df.fillna('nan')
#train=train.fillna('nan')


In [9]:
train.shape,test_df.shape

((5287, 2), (1962, 1))

In [10]:
import re
def clean_text(text):
    #Remove emojis and special chars
    clean=text
    #reg = re.compile(r'\\.+?(?=\\B|$)') # Using raw string and escaping backslashes if needed
    #clean = text.apply(lambda r: re.sub(reg, string=r, repl=''))
    #reg = re.compile(r'\\x89Û_')
    #clean = clean.apply(lambda r: re.sub(reg, string=r, repl=' ')) # Re-evaluating these if they become active
    reg = re.compile(r'&amp') # Corrected: removed unnecessary '\'
    clean = clean.apply(lambda r: re.sub(reg, string=r, repl='&'))
    reg = re.compile(r'\n') # Corrected: using raw string for newline character
    clean = clean.apply(lambda r: re.sub(reg, string=r, repl=' '))

    #Remove hashtag symbol (#)
    #clean = clean.apply(lambda r: r.replace('#', ''))

    #Remove user names
    reg = re.compile(r'@[a-zA-Z0-9_]+') # Corrected: removed unnecessary '\'
    clean = clean.apply(lambda r: re.sub(reg, string=r, repl='@'))

    #Remove URLs
    reg = re.compile(r'https?\S+(?=\s|$)') # Corrected: added 'r' for raw string
    clean = clean.apply(lambda r: re.sub(reg, string=r, repl='www'))

    #Lowercase
    #clean = clean.apply(lambda r: r.lower())
    return clean
train['text'] = clean_text(train['text']) # Changed 'safe_text' to 'text'
test_df['text'] = clean_text(test_df['text']) # Changed 'safe_text' to 'text'

In [11]:
train.head()

,text,target
0,The bitcoin halving is cancelled due to,1
1,MercyOfAllah In good times wrapped in its gran...,0
2,266 Days No Digital India No Murder of e learn...,1
3,India is likely to run out of the remaining RN...,1
4,In these tough times the best way to grow is t...,0


In [12]:
! pip uninstall -y fastai
! pip install pytorch-transformers
! pip install fastai==1.0.61

Found existing installation: fastai 1.0.61
Uninstalling fastai-1.0.61:
  Successfully uninstalled fastai-1.0.61
  Using cached fastai-1.0.61-py3-none-any.whl.metadata (14 kB)
Using cached fastai-1.0.61-py3-none-any.whl (239 kB)


In [13]:
from fastai.text import *
from fastai.text.transform import BaseTokenizer, Tokenizer, Vocab
from fastai.text.data import TextDataBunch, TextList, TokenizeProcessor, NumericalizeProcessor, FloatList, DatasetType
from fastai.basic_train import Learner
from fastai.metrics import *
from pytorch_transformers import RobertaTokenizer
from pathlib import Path
import torch

In [14]:
# Creating a config object to store task specific information
class Config(dict):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        for k, v in kwargs.items():
            setattr(self, k, v)

    def set(self, key, val):
        self[key] = val
        setattr(self, key, val)

config = Config(
    testing=False,
    seed = 2019,
    roberta_model_name='roberta-large', # can also be exchnaged with roberta-large
    max_lr=1e-5,
    epochs=2,
    use_fp16=False,
    bs=4,
    max_seq_len=96,
    num_labels = 3,
    hidden_dropout_prob=.05,
    hidden_size=1024, # 1024 for roberta-large
    start_tok = "<s>",
    end_tok = "</s>",
)

In [15]:
train.dtypes

,0
text,object
target,int64


In [16]:
class FastAiRobertaTokenizer(BaseTokenizer):
    """Wrapper around RobertaTokenizer to be compatible with fastai"""
    def __init__(self, tokenizer: RobertaTokenizer, max_seq_len: int=128, **kwargs):
        self._pretrained_tokenizer = tokenizer
        self.max_seq_len = max_seq_len
    def __call__(self, *args, **kwargs):
        return self
    def tokenizer(self, t:str) -> List[str]:
        """Adds Roberta bos and eos tokens and limits the maximum sequence length"""
        return [config.start_tok] + self._pretrained_tokenizer.tokenize(t)[:self.max_seq_len - 2] + [config.end_tok]

In [17]:
# create fastai tokenizer for roberta
roberta_tok = RobertaTokenizer.from_pretrained("roberta-large")

fastai_tokenizer = Tokenizer(tok_func=FastAiRobertaTokenizer(roberta_tok, max_seq_len=config.max_seq_len),
                             pre_rules=[], post_rules=[])

In [18]:
# create fastai vocabulary for roberta
path = Path()
roberta_tok.save_vocabulary(path)

with open('vocab.json', 'r') as f:
    roberta_vocab_dict = json.load(f)

fastai_roberta_vocab = Vocab(list(roberta_vocab_dict.keys()))

In [37]:
import numpy as np
# Setting up pre-processors
class RobertaTokenizeProcessor(TokenizeProcessor):
    def __init__(self, tokenizer):
         super().__init__(tokenizer=tokenizer, include_bos=False, include_eos=False)

class RobertaNumericalizeProcessor(NumericalizeProcessor):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def process_one(self, item):
        # Convert tokens to numerical IDs using the vocabulary, which returns a List[int]
        numericalized_list = super().process_one(item)

        # Convert the list to a 1D NumPy array of the correct dtype
        numericalized_array = np.array(numericalized_list, dtype=np.int64).flatten()

        current_len = len(numericalized_array)

        # Pad or truncate the numericalized sequence to a fixed length (config.max_seq_len)
        if current_len < config.max_seq_len:
            padded_item = np.pad(numericalized_array, (0, config.max_seq_len - current_len), 'constant', constant_values=0)
        else:
            padded_item = numericalized_array[:config.max_seq_len]

        # Highly defensive check: Ensure the final output is of the exact expected length and dtype
        if len(padded_item) != config.max_seq_len:
            # If, for any unexpected reason, the length is not correct, force it
            temp_padded = np.zeros(config.max_seq_len, dtype=np.int64)
            copy_len = min(config.max_seq_len, len(padded_item))
            temp_padded[:copy_len] = padded_item[:copy_len]
            padded_item = temp_padded

        return padded_item.astype(np.int64) # Returns a 1D numpy array of length config.max_seq_len and consistent dtype

def get_roberta_processor(tokenizer:Tokenizer=None, vocab:Vocab=None):
    """
    Constructing preprocessors for Roberta
    We remove sos and eos tokens since we add that ourselves in the tokenizer.
    We also use a custom vocabulary to match the numericalization with the original Roberta model.
    """
    return [RobertaTokenizeProcessor(tokenizer=tokenizer), RobertaNumericalizeProcessor(vocab=vocab)]


In [30]:
# Creating a Roberta specific DataBunch class
class RobertaDataBunch(TextDataBunch):
    "Create a `TextDataBunch` suitable for training Roberta"
    @classmethod
    def create(cls, train_ds, valid_ds, test_ds=None, path:PathOrStr='.', bs:int=64, val_bs:int=None, pad_idx=1,
               pad_first=True, device:torch.device=None, no_check:bool=False, backwards:bool=False,
               dl_tfms:Optional[Collection[Callable]]=None, **dl_kwargs) -> DataBunch:
        "Function that transform the `datasets` in a `DataBunch` for classification. Passes `**dl_kwargs` on to `DataLoader()`"
        datasets = cls._init_ds(train_ds, valid_ds, test_ds)
        val_bs = ifnone(val_bs, bs)
        collate_fn = partial(pad_collate, pad_idx=pad_idx, pad_first=pad_first, backwards=backwards)
        train_sampler = SortishSampler(datasets[0].x, key=lambda t: len(datasets[0][t][0].data), bs=bs)
        train_dl = DataLoader(datasets[0], batch_size=bs, sampler=train_sampler, drop_last=True, **dl_kwargs)
        dataloaders = [train_dl]
        for ds in datasets[1:]:
            lengths = [len(t) for t in ds.x.items]
            sampler = SortSampler(ds.x, key=lengths.__getitem__)
            dataloaders.append(DataLoader(ds, batch_size=val_bs, sampler=sampler, **dl_kwargs))
        return cls(*dataloaders, path=path, device=device, dl_tfms=dl_tfms, collate_fn=collate_fn, no_check=no_check)


In [31]:
class RobertaTextList(TextList):
    _bunch = RobertaDataBunch
    _label_cls = TextList

In [32]:
test_df.shape

(1962, 1)

In [33]:
sub=pd.read_csv('/content/drive/My Drive/csv/SampleSubmission.csv')


In [38]:
processor = get_roberta_processor(tokenizer=fastai_tokenizer, vocab=fastai_roberta_vocab)

# Convert labels to float explicitly to ensure consistency for FloatList
train[label_cols] = train[label_cols].astype(float)

data = RobertaTextList.from_df(train, ".", cols=feat_cols, processor=processor) \
    .split_by_rand_pct(seed=2021) \
    .label_from_df(cols=label_cols,label_cls=FloatList) \
    .add_test(RobertaTextList.from_df(test_df, ".", cols=feat_cols, processor=processor)) \
    .databunch(bs=4, pad_first=False, pad_idx=0)


import torch
import torch.nn as nn
from pytorch_transformers import RobertaModel

# defining our model architecture
class CustomRobertaModel(nn.Module):
    def __init__(self,num_labels=1):
        super(CustomRobertaModel,self).__init__()
        self.num_labels = num_labels
        self.roberta = RobertaModel.from_pretrained(config.roberta_model_name)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, num_labels) # defining final output layer

    def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None):
        _ , pooled_output = self.roberta(input_ids, token_type_ids, attention_mask) #
        logits = self.classifier(pooled_output)
        return logits
roberta_model = CustomRobertaModel()

learn = Learner(data, roberta_model, metrics=[rmse])

learn.model.roberta.train() # setting roberta to train as it is in eval mode by default
learn.fit_one_cycle(3, max_lr=config.max_lr)


def get_preds_as_nparray(ds_type) -> np.ndarray:
    learn.model.roberta.eval()
    preds = learn.get_preds(ds_type)[0].detach().cpu().numpy()
    sampler = [i for i in data.dl(ds_type).sampler]
    reverse_sampler = np.argsort(sampler)
    ordered_preds = preds[reverse_sampler, :]
    # The model outputs a single value for regression, so ordered_preds will have shape (N, 1).
    # np.argmax is for classification, but here it would just return an array of zeros.
    # We need the actual predicted float values.
    pred_values = ordered_preds.flatten() # Get the actual predicted float values
    return ordered_preds, pred_values

test_preds,preds = get_preds_as_nparray(DatasetType.Test)
sub['label']=preds

for e,p in enumerate(sub['label']):
  if p<-1:
    sub['label'][e]=-1

for e,p in enumerate(sub['label']):
  if p>1:
    sub['label'][e]=1

sub.to_csv('sentimentsub1.csv',index=False)


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (1057,) + inhomogeneous part.

In [ ]:
processor = get_roberta_processor(tokenizer=fastai_tokenizer, vocab=fastai_roberta_vocab)
data = RobertaTextList.from_df(train, ".", cols=feat_cols, processor=processor) \
    .split_by_rand_pct(seed=30000) \
    .label_from_df(cols=label_cols,label_cls=FloatList) \
    .add_test(RobertaTextList.from_df(test_df, ".", cols=feat_cols, processor=processor)) \
    .databunch(bs=4, pad_first=False, pad_idx=0)


import torch
import torch.nn as nn
from pytorch_transformers import RobertaModel

# defining our model architecture
class CustomRobertaModel(nn.Module):
    def __init__(self,num_labels=1):
        super(CustomRobertaModel,self).__init__()
        self.num_labels = num_labels
        self.roberta = RobertaModel.from_pretrained(config.roberta_model_name)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, num_labels) # defining final output layer

    def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None):
        _ , pooled_output = self.roberta(input_ids, token_type_ids, attention_mask) #
        logits = self.classifier(pooled_output)
        return logits
roberta_model = CustomRobertaModel()

learn = Learner(data, roberta_model, metrics=[rmse])

learn.model.roberta.train() # setting roberta to train as it is in eval mode by default
learn.fit_one_cycle(3, max_lr=config.max_lr)


def get_preds_as_nparray(ds_type) -> np.ndarray:
    learn.model.roberta.eval()
    preds = learn.get_preds(ds_type)[0].detach().cpu().numpy()
    sampler = [i for i in data.dl(ds_type).sampler]
    reverse_sampler = np.argsort(sampler)
    ordered_preds = preds[reverse_sampler, :]
    pred_values = np.argmax(ordered_preds, axis=1)
    return ordered_preds, pred_values

test_preds2,preds = get_preds_as_nparray(DatasetType.Test)
sub['label']=test_preds2

for e,p in enumerate(sub['label']):
  if p<-1:
    sub['label'][e]=-1

for e,p in enumerate(sub['label']):
  if p>1:
    sub['label'][e]=1

sub.to_csv('sentimentsub2.csv',index=False)


In [ ]:
processor = get_roberta_processor(tokenizer=fastai_tokenizer, vocab=fastai_roberta_vocab)
data = RobertaTextList.from_df(train, ".", cols=feat_cols, processor=processor) \
    .split_by_rand_pct(seed=42) \
    .label_from_df(cols=label_cols,label_cls=FloatList) \
    .add_test(RobertaTextList.from_df(test_df, ".", cols=feat_cols, processor=processor)) \
    .databunch(bs=4, pad_first=False, pad_idx=0)


import torch
import torch.nn as nn
from pytorch_transformers import RobertaModel

# defining our model architecture
class CustomRobertaModel(nn.Module):
    def __init__(self,num_labels=1):
        super(CustomRobertaModel,self).__init__()
        self.num_labels = num_labels
        self.roberta = RobertaModel.from_pretrained(config.roberta_model_name)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, num_labels) # defining final output layer

    def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None):
        _ , pooled_output = self.roberta(input_ids, token_type_ids, attention_mask) #
        logits = self.classifier(pooled_output)
        return logits
roberta_model = CustomRobertaModel()

learn = Learner(data, roberta_model, metrics=[rmse])

learn.model.roberta.train() # setting roberta to train as it is in eval mode by default
learn.fit_one_cycle(3, max_lr=config.max_lr)


def get_preds_as_nparray(ds_type) -> np.ndarray:
    learn.model.roberta.eval()
    preds = learn.get_preds(ds_type)[0].detach().cpu().numpy()
    sampler = [i for i in data.dl(ds_type).sampler]
    reverse_sampler = np.argsort(sampler)
    ordered_preds = preds[reverse_sampler, :]
    pred_values = np.argmax(ordered_preds, axis=1)
    return ordered_preds, pred_values

test_preds3,preds = get_preds_as_nparray(DatasetType.Test)
sub['label']=test_preds3

for e,p in enumerate(sub['label']):
  if p<-1:
    sub['label'][e]=-1

for e,p in enumerate(sub['label']):
  if p>1:
    sub['label'][e]=1

sub.to_csv('sentimentsub3.csv',index=False)

In [ ]:
sub1=pd.read_csv('/content/sentimentsub1.csv')
sub2=pd.read_csv('/content/sentimentsub2.csv')
sub3=pd.read_csv('/content/sentimentsub3.csv')
sub1['label']=0.35*sub1['label']+0.3*sub2['label']+0.35*sub3['label']
sub1.to_csv('sentimentsub_ens.csv',index=False)#0.5040

In [ ]:
processor = get_roberta_processor(tokenizer=fastai_tokenizer, vocab=fastai_roberta_vocab)
data = RobertaTextList.from_df(train, ".", cols=feat_cols, processor=processor) \
    .split_by_rand_pct(seed=2020) \
    .label_from_df(cols=label_cols,label_cls=FloatList) \
    .add_test(RobertaTextList.from_df(test_df, ".", cols=feat_cols, processor=processor)) \
    .databunch(bs=4, pad_first=False, pad_idx=0)


import torch
import torch.nn as nn
from pytorch_transformers import RobertaModel

# defining our model architecture
class CustomRobertaModel(nn.Module):
    def __init__(self,num_labels=1):
        super(CustomRobertaModel,self).__init__()
        self.num_labels = num_labels
        self.roberta = RobertaModel.from_pretrained(config.roberta_model_name)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, num_labels) # defining final output layer

    def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None):
        _ , pooled_output = self.roberta(input_ids, token_type_ids, attention_mask) #
        logits = self.classifier(pooled_output)
        return logits
roberta_model = CustomRobertaModel()

learn = Learner(data, roberta_model, metrics=[rmse])

learn.model.roberta.train() # setting roberta to train as it is in eval mode by default
learn.fit_one_cycle(3, max_lr=config.max_lr)


def get_preds_as_nparray(ds_type) -> np.ndarray:
    learn.model.roberta.eval()
    preds = learn.get_preds(ds_type)[0].detach().cpu().numpy()
    sampler = [i for i in data.dl(ds_type).sampler]
    reverse_sampler = np.argsort(sampler)
    ordered_preds = preds[reverse_sampler, :]
    pred_values = np.argmax(ordered_preds, axis=1)
    return ordered_preds, pred_values

test_preds4,preds = get_preds_as_nparray(DatasetType.Test)
sub['label']=test_preds4

for e,p in enumerate(sub['label']):
  if p<-1:
    sub['label'][e]=-1

for e,p in enumerate(sub['label']):
  if p>1:
    sub['label'][e]=1

sub.to_csv('sentimentsub4.csv',index=False)

In [ ]:
processor = get_roberta_processor(tokenizer=fastai_tokenizer, vocab=fastai_roberta_vocab)
data = RobertaTextList.from_df(train, ".", cols=feat_cols, processor=processor) \
    .split_by_rand_pct(seed=80000) \
    .label_from_df(cols=label_cols,label_cls=FloatList) \
    .add_test(RobertaTextList.from_df(test_df, ".", cols=feat_cols, processor=processor)) \
    .databunch(bs=4, pad_first=False, pad_idx=0)


import torch
import torch.nn as nn
from pytorch_transformers import RobertaModel

# defining our model architecture
class CustomRobertaModel(nn.Module):
    def __init__(self,num_labels=1):
        super(CustomRobertaModel,self).__init__()
        self.num_labels = num_labels
        self.roberta = RobertaModel.from_pretrained(config.roberta_model_name)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, num_labels) # defining final output layer

    def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None):
        _ , pooled_output = self.roberta(input_ids, token_type_ids, attention_mask) #
        logits = self.classifier(pooled_output)
        return logits
roberta_model = CustomRobertaModel()

learn = Learner(data, roberta_model, metrics=[rmse])

learn.model.roberta.train() # setting roberta to train as it is in eval mode by default
learn.fit_one_cycle(3, max_lr=config.max_lr)


def get_preds_as_nparray(ds_type) -> np.ndarray:
    learn.model.roberta.eval()
    preds = learn.get_preds(ds_type)[0].detach().cpu().numpy()
    sampler = [i for i in data.dl(ds_type).sampler]
    reverse_sampler = np.argsort(sampler)
    ordered_preds = preds[reverse_sampler, :]
    pred_values = np.argmax(ordered_preds, axis=1)
    return ordered_preds, pred_values

test_preds5,preds = get_preds_as_nparray(DatasetType.Test)
sub['label']=test_preds5

for e,p in enumerate(sub['label']):
  if p<-1:
    sub['label'][e]=-1

for e,p in enumerate(sub['label']):
  if p>1:
    sub['label'][e]=1

sub.to_csv('sentimentsub5.csv',index=False)

In [ ]:
processor = get_roberta_processor(tokenizer=fastai_tokenizer, vocab=fastai_roberta_vocab)
data = RobertaTextList.from_df(train, ".", cols=feat_cols, processor=processor) \
    .split_by_rand_pct(seed=666) \
    .label_from_df(cols=label_cols,label_cls=FloatList) \
    .add_test(RobertaTextList.from_df(test_df, ".", cols=feat_cols, processor=processor)) \
    .databunch(bs=4, pad_first=False, pad_idx=0)


import torch
import torch.nn as nn
from pytorch_transformers import RobertaModel

# defining our model architecture
class CustomRobertaModel(nn.Module):
    def __init__(self,num_labels=1):
        super(CustomRobertaModel,self).__init__()
        self.num_labels = num_labels
        self.roberta = RobertaModel.from_pretrained(config.roberta_model_name)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, num_labels) # defining final output layer

    def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None):
        _ , pooled_output = self.roberta(input_ids, token_type_ids, attention_mask) #
        logits = self.classifier(pooled_output)
        return logits
roberta_model = CustomRobertaModel()

learn = Learner(data, roberta_model, metrics=[rmse])

learn.model.roberta.train() # setting roberta to train as it is in eval mode by default
learn.fit_one_cycle(3, max_lr=config.max_lr)


def get_preds_as_nparray(ds_type) -> np.ndarray:
    learn.model.roberta.eval()
    preds = learn.get_preds(ds_type)[0].detach().cpu().numpy()
    sampler = [i for i in data.dl(ds_type).sampler]
    reverse_sampler = np.argsort(sampler)
    ordered_preds = preds[reverse_sampler, :]
    pred_values = np.argmax(ordered_preds, axis=1)
    return ordered_preds, pred_values

test_preds6,preds = get_preds_as_nparray(DatasetType.Test)
sub['label']=test_preds6

for e,p in enumerate(sub['label']):
  if p<-1:
    sub['label'][e]=-1

for e,p in enumerate(sub['label']):
  if p>1:
    sub['label'][e]=1

sub.to_csv('sentimentsub6.csv',index=False)

In [ ]:
sub4=pd.read_csv('/content/sentimentsub4.csv')
sub5=pd.read_csv('/content/sentimentsub5.csv')
sub6=pd.read_csv('/content/sentimentsub6.csv')
sub4['label']=0.3*sub4['label']+0.3*sub5['label']+0.4*sub6['label']
sub4.to_csv('sentimentsub_ens1.csv',index=False)#0.50120

In [ ]:
sub11=pd.read_csv('/content/sentimentsub_ens.csv')
sub22=pd.read_csv('/content/sentimentsub_ens1.csv')
sub11['label']=0.5*sub11['label']+0.5*sub22['label']
sub11.to_csv('final_sub.csv',index=False)#0.49818

In [ ]:
#Lstm approach

In [ ]:
from tqdm import tqdm
from sklearn.svm import SVC
from keras.models import Sequential
from keras.layers.recurrent import LSTM, GRU
from keras.layers.core import Dense, Activation, Dropout
from keras.layers.embeddings import Embedding
from keras.layers.normalization import BatchNormalization
from keras.utils import np_utils
from sklearn import preprocessing, decomposition, model_selection, metrics, pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from keras.layers import GlobalMaxPooling1D, Conv1D, MaxPooling1D, Flatten, Bidirectional, SpatialDropout1D
from keras.preprocessing import sequence, text
from keras.callbacks import EarlyStopping
from nltk import word_tokenize
from nltk.corpus import stopwords
# Setup
!pip install -q wordcloud
import wordcloud
from keras.layers import GlobalMaxPooling1D, Conv1D, MaxPooling1D, Flatten, Bidirectional, SpatialDropout1D
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
import pandas as pd
from keras.layers.convolutional import Conv1D
from keras.layers.convolutional import MaxPooling1D
from keras.optimizers import RMSprop,SGD
from keras.preprocessing.text import Tokenizer
from keras.preprocessing import sequence
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping

from keras.models import Model
from keras.layers import LSTM, Activation, Dense, Dropout, Input, Embedding
df=pd.read_csv('/content/Train (10).csv')

df = df.loc[df['agreement'] > 0.34]
df=df.dropna()
Y = df.label

# Print initial shape
print(Y.shape)
X=df.safe_text


max_len = 200
# Build the dictionary of indexes
tok = Tokenizer()
tok.fit_on_texts(X)
# Change texts into sequence of indexes
sequences = tok.texts_to_sequences(X)
# Pad the sequences
sequences_matrix_train = sequence.pad_sequences(sequences,maxlen=max_len)



test=pd.read_csv('/content/Test (5).csv')
test=test.safe_text
test=test.fillna(method='ffill')
#same preprocessing for test

tok.fit_on_texts(test)
sequences = tok.texts_to_sequences(test)
sequences_matrix_test = sequence.pad_sequences(sequences,maxlen=max_len)

In [ ]:
#choosing vocabulary for embedding layer
max_words=2000000
def RNN():
    inputs = Input(name='inputs',shape=[max_len])
    layer = Embedding(max_words,50,input_length=max_len)(inputs)
    layer = Dropout(0.5)(layer)
    layer = (Conv1D(filters=64,
                 kernel_size=4,
                 padding='valid',
                 activation='sigmoid',
                 strides=1))(layer)
    layer = (MaxPooling1D(pool_size=2))(layer)
    layer = Dropout(0.5)(layer)
    layer = LSTM(64)(layer)


    layer = Dense(256,name='FC1')(layer)
    layer = Activation('relu')(layer)
    layer = Dropout(0.5)(layer)
    layer = Dense(1,name='out_layer')(layer)
    layer = Activation('tanh')(layer)
    model = Model(inputs=inputs,outputs=layer)
    return model
model = RNN()
model.summary()
model.compile(loss='mean_squared_error',optimizer='adam',metrics=['accuracy'])

In [ ]:
model.fit(sequences_matrix_train,Y,batch_size=128,epochs=8,
          validation_split=0.2,callbacks=[EarlyStopping(monitor='val_loss',min_delta=0.0001)])

In [ ]:
p=model.predict(sequences_matrix_test)


In [ ]:
p

In [ ]:
#choosing vocabulary for embedding layer
max_words=2000000
def RNN():
    inputs = Input(name='inputs',shape=[max_len])
    layer = Embedding(max_words,50,input_length=max_len)(inputs)


    layer = LSTM(64)(layer)


    layer = Dense(256,name='FC1')(layer)
    layer = Activation('relu')(layer)
    layer = Dropout(0.5)(layer)
    layer = Dense(1,name='out_layer')(layer)
    layer = Activation('tanh')(layer)
    model = Model(inputs=inputs,outputs=layer)
    return model
model = RNN()
model.summary()
model.compile(loss='mean_squared_error',optimizer='adam',metrics=['accuracy'])

In [ ]:
model.fit(sequences_matrix_train,Y,batch_size=128,epochs=2,
          validation_split=0.2,callbacks=[EarlyStopping(monitor='val_loss',min_delta=0.0001)])

In [ ]:
q=model.predict(sequences_matrix_test)
a=q*2/3+p*1/3

In [ ]:
sub=pd.read_csv('/content/SampleSubmission (11).csv')
sub['label']=a
final=pd.read_csv('/content/final_sub.csv')
final['label']=final['label']*0.85+sub['label']*0.15
final.to_csv('se.csv',index=False)

In [ ]:
#0.487578702571676